In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ravikantsaraf/assesment/eval_finetuned_only.csv


In [2]:
%%capture
!pip install unsloth unsloth_zoo wandb
!pip install -q rouge-score

In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
import pandas as pd, json, torch

SYSTEM_PROMPT = "You are a clinical assistant. Given an unstructured doctor's note, generate a structured SOAP summary."

def generate_soap(mdl, tok, note):
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{note}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
# Upload your eval_finetuned_only.csv to Kaggle first
eval_df = pd.read_csv('/kaggle/input/datasets/ravikantsaraf/assesment/eval_finetuned_only.csv')
print(f"Loaded {len(eval_df)} samples")
print(eval_df.columns.tolist())  # confirm columns

Loaded 50 samples
['raw_note', 'soap', 'text', 'finetuned']


In [5]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_outputs = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    base_outputs.append(generate_soap(base_model, base_tokenizer, row['raw_note']))

eval_df['base'] = base_outputs
print("Base model done!")

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


  0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Base model done!


In [6]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'ROUGE-1': round(sum(r1)/len(r1), 4),
        'ROUGE-2': round(sum(r2)/len(r2), 4),
        'ROUGE-L': round(sum(rL)/len(rL), 4),
    }

refs = eval_df['soap'].tolist()
base_rouge = compute_rouge(eval_df['base'].tolist(), refs)
ft_rouge   = compute_rouge(eval_df['finetuned'].tolist(), refs)

print("\n=== ROUGE SCORES ===")
print(f"{'Metric':<12} {'Base':>8} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 44)
for m in ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']:
    delta = f"+{round((ft_rouge[m] - base_rouge[m]) * 100, 1)}%"
    print(f"{m:<12} {base_rouge[m]:>8} {ft_rouge[m]:>12} {delta:>8}")


=== ROUGE SCORES ===
Metric           Base   Fine-tuned    Delta
--------------------------------------------
ROUGE-1        0.5519       0.7036   +15.2%
ROUGE-2        0.2929       0.4298   +13.7%
ROUGE-L         0.368       0.5175   +14.9%


In [7]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'ROUGE-1': round(sum(r1)/len(r1), 4),
        'ROUGE-2': round(sum(r2)/len(r2), 4),
        'ROUGE-L': round(sum(rL)/len(rL), 4),
    }

refs = eval_df['soap'].tolist()
base_rouge = compute_rouge(eval_df['base'].tolist(), refs)
ft_rouge   = compute_rouge(eval_df['finetuned'].tolist(), refs)

print("\n=== ROUGE SCORES ===")
print(f"{'Metric':<12} {'Base':>8} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 44)
for m in ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']:
    delta = f"+{round((ft_rouge[m] - base_rouge[m]) * 100, 1)}%"
    print(f"{m:<12} {base_rouge[m]:>8} {ft_rouge[m]:>12} {delta:>8}")


=== ROUGE SCORES ===
Metric           Base   Fine-tuned    Delta
--------------------------------------------
ROUGE-1        0.5519       0.7036   +15.2%
ROUGE-2        0.2929       0.4298   +13.7%
ROUGE-L         0.368       0.5175   +14.9%


In [ ]:
eval_df.to_csv('/kaggle/working/eval_final.csv', index=False)

scores = {'base': base_rouge, 'finetuned': ft_rouge}
with open('/kaggle/working/scores.json', 'w') as f:
    json.dump(scores, f, indent=2)

print(json.dumps(scores, indent=2))
print("\nDownload eval_final.csv and scores.json from Output tab")